# Build Dataset

Build the harmonized `kaya-go/moku-v1` dataset from raw COCO datasets.

**Sources:**

- `Go Game detection.v10i.coco` — 256 images (board, black/white stones, empty intersections)
- `go-chess 2.v3-go-chess.v1.coco` — 236 images (board, black/white stones)

**Harmonized categories:**
| ID | Name | Description |
|-----|------|------|
| 0 | board | Full Go board bounding box |
| 1 | black_stone | Individual black stone |
| 2 | white_stone | Individual white stone |

See [docs/dataset.md](../docs/dataset.md) for full details.


In [1]:
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
from PIL import Image

from moku.dataset import (
    CATEGORIES,
    ID_TO_CATEGORY,
    build_dataset,
    compute_split_stats,
    load_coco_dataset,
    get_source_categories,
)

In [2]:
RAW_DATA_DIR = Path("/Users/hadim/Data/moku/raw")

DATASET_DIRS = {
    "go_game_v10": RAW_DATA_DIR / "Go Game detection.v10i.coco",
    "go_chess": RAW_DATA_DIR / "go-chess 2.v3-go-chess.v1.coco",
}

# Verify datasets exist
for name, path in DATASET_DIRS.items():
    exists = path.exists()
    print(f"{'OK' if exists else 'MISSING'} {name}: {path.name}")

OK go_game_v10: Go Game detection.v10i.coco
OK go_chess: go-chess 2.v3-go-chess.v1.coco


## Explore Source Categories

Compare the original categories across the raw datasets before harmonization.


In [3]:
for name, path in DATASET_DIRS.items():
    coco_splits = load_coco_dataset(path)
    train_data = coco_splits.get("train", {})
    cats = get_source_categories(train_data)

    print(f"\n{name}:")
    for cid, cname in sorted(cats.items()):
        print(f"  {cid}: {cname}")

    # Count images per split
    for split_name, split_data in coco_splits.items():
        n_imgs = len(split_data.get("images", []))
        n_anns = len(split_data.get("annotations", []))
        print(f"  {split_name}: {n_imgs} images, {n_anns:,} annotations")


go_game_v10:
  0: go-game
  1: black_stone
  2: board
  3: board_corner
  4: empty
  5: empty_corner
  6: empty_edge
  7: white_stone
  train: 241 images, 85,751 annotations
  valid: 14 images, 4,748 annotations
  test: 1 images, 364 annotations

go_chess:
  0: go-stone-FMbU
  1: black_stone
  2: goboard
  3: white_stone
  train: 206 images, 13,079 annotations
  valid: 20 images, 1,283 annotations
  test: 10 images, 558 annotations


## Build Harmonized Dataset

Load raw COCO datasets, apply category harmonization (keep only board, black_stone, white_stone), and create a unified HuggingFace DatasetDict.


In [4]:
dataset = build_dataset(RAW_DATA_DIR)
dataset

  go_game_v10: 249 images
  go_chess: 236 images
  Total pooled: 485 images
  train: 382 images (153 base) [go_chess: 195, go_game_v10: 187]
  validation: 53 images (20 base) [go_chess: 25, go_game_v10: 28]
  test: 50 images (20 base) [go_chess: 16, go_game_v10: 34]


DatasetDict({
    train: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 382
    })
    validation: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 53
    })
    test: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 50
    })
})

## Dataset Statistics

Analyze the harmonized dataset: images per split, objects per category, source distribution.


In [5]:
import pandas as pd

rows = []
for split_name, split_ds in dataset.items():
    stats = compute_split_stats(split_ds)

    # Summary row
    rows.append(
        {
            "split": split_name,
            "images": stats["num_images"],
            "total_objects": stats["total_objects"],
            "avg_objects_per_image": round(stats["avg_objects_per_image"], 1),
        }
    )

summary_df = pd.DataFrame(rows)
display(summary_df)

# Per-source breakdown
source_rows = []
for split_name, split_ds in dataset.items():
    stats = compute_split_stats(split_ds)
    for source, count in stats["source_counts"].most_common():
        source_rows.append({"split": split_name, "source": source, "images": count})

source_df = pd.DataFrame(source_rows)
display(source_df)

# Per-category breakdown
cat_rows = []
for split_name, split_ds in dataset.items():
    stats = compute_split_stats(split_ds)
    for cat_id, count in stats["category_counts"].most_common():
        cat_rows.append({"split": split_name, "category": ID_TO_CATEGORY[cat_id], "count": count})

cat_df = pd.DataFrame(cat_rows)
display(cat_df)

,split,images,total_objects,avg_objects_per_image
0,train,382,24648,64.5
1,validation,53,2220,41.9
2,test,50,3029,60.6


,split,source,images
0,train,go_chess,195
1,train,go_game_v10,187
2,validation,go_game_v10,28
3,validation,go_chess,25
4,test,go_game_v10,34
5,test,go_chess,16


,split,category,count
0,train,black_stone,12902
1,train,white_stone,11364
2,train,board,382
3,validation,black_stone,1205
4,validation,white_stone,962
5,validation,board,53
6,test,black_stone,1602
7,test,white_stone,1377
8,test,board,50


## Browse Dataset

Interactive browser to navigate samples and inspect annotations.

In [7]:
from moku.viz import browse_dataset

browse_dataset(dataset)

## Push to Hugging Face Hub

Upload the harmonized dataset to `kaya-go/moku-v1`.


In [8]:
HF_DATASET_REPO = "kaya-go/moku-v1"

dataset.push_to_hub(HF_DATASET_REPO, private=False)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/382 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/862 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/kaya-go/moku-v1/commit/9b78a39c50f52ef7df9f523fed7ef1e6fb571afb', commit_message='Upload dataset', commit_description='', oid='9b78a39c50f52ef7df9f523fed7ef1e6fb571afb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/kaya-go/moku-v1', endpoint='https://huggingface.co', repo_type='dataset', repo_id='kaya-go/moku-v1'), pr_revision=None, pr_num=None)